# MLJAR 与 TabularX 完整工具包对比

**数据集：Wine（scikit-learn load_wine（源自 UCI Wine），三分类）**。Wine 数据集包含 178 个葡萄酒样本的 13 项化学测量特征，目标是判断样本属于三个葡萄酒品种中的哪一类。

双方使用同一开发集、同一外部测试集、随机种子 42，以及 Random Forest、XGBoost、LightGBM、CatBoost。外部测试集只在模型锁定后评估一次。性能指标只描述本次固定配置；核心比较对象是实际生成的解释性与可信性工件。


In [1]:
from pathlib import Path
import os, sys, json, hashlib, time, subprocess
import joblib
import numpy as np
import pandas as pd
import openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import tabularx as tbx
from supervised.automl import AutoML

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (ROOT, *ROOT.parents) if (p / "TabularX-project").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("未找到项目根目录")
DATASET_KEY = "wine"
DATASET_TITLE = "Wine"
TASK = "classification"
TASK_CN = "三分类"
DESCRIPTION = 'Wine 数据集包含 178 个葡萄酒样本的 13 项化学测量特征，目标是判断样本属于三个葡萄酒品种中的哪一类。'
SOURCE = 'scikit-learn load_wine（源自 UCI Wine）'
OUTPUT = PROJECT_ROOT / "benchmark-output" / DATASET_KEY
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / "matplotlib-cache"))
os.environ.setdefault("JOBLIB_TEMP_FOLDER", str(OUTPUT / "joblib-temp"))
SEED = 42
ALGORITHMS_TBX = ["random_forest", "xgboost", "lightgbm", "catboost"]
ALGORITHMS_MLJAR = ["Random Forest", "Xgboost", "LightGBM", "CatBoost"]
openml.config.set_root_cache_directory(str(OUTPUT / "openml-cache"))
print("数据集:", DATASET_TITLE, "| 任务:", TASK_CN, "| 输出:", OUTPUT)


D:\anaconda\envs\tabularx-benchmark\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据集: Wine | 任务: 三分类 | 输出: C:\Users\<user>\Documents\ML26-qyr-develop\benchmark-output\wine


## 1. 数据、敏感属性与共同外部测试集

In [2]:
from sklearn.datasets import load_wine
bundle = load_wine(as_frame=True)
X = bundle.data.reset_index(drop=True)
y_raw = bundle.target.reset_index(drop=True)
X_tbx = X.copy()
X_model = X.copy()
S = None
forbidden_features = []
sensitive_columns = []
privileged_value = underprivileged_value = None
source_id = {"sklearn_loader": "load_wine", "target": "target"}

if TASK == "classification":
    encoder = LabelEncoder()
    y = pd.Series(encoder.fit_transform(y_raw.astype("string")), name="target")
    label_mapping = {str(label): int(index) for index, label in enumerate(encoder.classes_)}
    stratify = y
else:
    y = pd.Series(y_raw, name="target", dtype=float)
    label_mapping = None
    stratify = None

indices = np.arange(len(X))
dev_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=SEED, stratify=stratify)
X_dev_tbx = X_tbx.iloc[dev_idx].reset_index(drop=True)
X_test_tbx = X_tbx.iloc[test_idx].reset_index(drop=True)
X_dev = X_model.iloc[dev_idx].reset_index(drop=True)
X_test = X_model.iloc[test_idx].reset_index(drop=True)
y_dev = y.iloc[dev_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
S_dev = None if S is None else S.iloc[dev_idx].reset_index(drop=True)

evidence_dir = OUTPUT / "shared-evidence"
evidence_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(evidence_dir / "split_indices.npz", dev=dev_idx, test=test_idx)
fingerprint_frame = pd.concat([X.astype("string"), pd.Series(y_raw).astype("string").rename("target_raw")], axis=1)
fingerprint = hashlib.sha256(pd.util.hash_pandas_object(fingerprint_frame, index=True).values.tobytes()).hexdigest()
manifest = {**source_id, "dataset": DATASET_TITLE, "task": TASK_CN, "rows": len(X),
            "original_features": X.shape[1], "model_input_features": X_model.shape[1],
            "dev_rows": len(dev_idx), "test_rows": len(test_idx), "seed": SEED,
            "dataset_sha256": fingerprint, "label_mapping": label_mapping,
            "forbidden_features": forbidden_features, "sensitive_columns": sensitive_columns}
(evidence_dir / "dataset_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([manifest]))


,sklearn_loader,target,dataset,task,rows,original_features,model_input_features,dev_rows,test_rows,seed,dataset_sha256,label_mapping,forbidden_features,sensitive_columns
0,load_wine,target,Wine,三分类,178,13,13,142,36,42,081f9c5ba314b9e76427d2fdfaca4d26d58051b52464fb...,"{'0': 0, '1': 1, '2': 2}",[],[]


## 2. 运行 TabularX

In [3]:
tbx_dir = OUTPUT / "tabularx"
ae = tbx.AutoEstimator(task=TASK, output_dir=tbx_dir, pdf_report=False)
ae.param.model_search.set_param("models", ALGORITHMS_TBX)
ae.param.model_search.set_param("n_trials", 3)
ae.param.model_search.set_param("n_jobs", 1)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.model_search.set_param("random_state", SEED)
ae.param.split.set_param("random_state", SEED)
ae.param.profile.forbidden_features = forbidden_features
ae.param.preprocess.apply_audit_drops = "high"

started = time.time()
if sensitive_columns:
    ae.fit(X_dev_tbx, y_dev, sensitive_features=sensitive_columns)
else:
    ae.fit(X_dev_tbx, y_dev)
tbx_seconds = time.time() - started
tbx_run_dir = Path(ae.output_dir)
tbx_pipeline = joblib.load(tbx_run_dir / "pipeline.joblib")
tbx_pred = np.asarray(tbx_pipeline.predict(X_test_tbx))
tbx_prob = np.asarray(tbx_pipeline.predict_proba(X_test_tbx)) if TASK == "classification" else None
print("TabularX 最佳模型:", ae.result.searched.best_model_name)
print("TabularX 运行秒数:", round(tbx_seconds, 2))
print("TabularX HTML:", ae.result.report_paths["full"])


[I 2026-09-29 16:33:56,021] A new study created in memory with name: random_forest_multiclass


TabularX 最佳模型: random_forest
TabularX 运行秒数: 19.38
TabularX HTML: C:\Users\<user>\Documents\ML26-qyr-develop\benchmark-output\wine\tabularx\runs\80eb834660224d93be7569ea776299d4\report.html


## 3. 运行 MLJAR

In [4]:
mljar_dir = OUTPUT / "mljar"
mljar_existed = (mljar_dir / "README.md").exists()
kwargs = dict(mode="Explain", results_path=str(mljar_dir), algorithms=ALGORITHMS_MLJAR,
              total_time_limit=600, train_ensemble=True, random_state=SEED, explain_level=2,
              eval_metric="auc" if TASK == "classification" and y.nunique() == 2 else
                          ("logloss" if TASK == "classification" else "rmse"))
if sensitive_columns and y.nunique() == 2:
    kwargs.update(fairness_metric="demographic_parity_ratio", fairness_threshold=0.8,
                  privileged_groups=[{sensitive_columns[0]: privileged_value}],
                  underprivileged_groups=[{sensitive_columns[0]: underprivileged_value}])
automl = AutoML(**kwargs)
started = time.time()
if sensitive_columns and y.nunique() == 2:
    automl.fit(X_dev, y_dev, sensitive_features=S_dev)
else:
    automl.fit(X_dev, y_dev)
mljar_observed_seconds = time.time() - started
runtime_file = mljar_dir / "benchmark_runtime.json"
if not mljar_existed or mljar_observed_seconds > 1:
    mljar_seconds = mljar_observed_seconds
    runtime_file.write_text(json.dumps({"first_full_run_seconds": mljar_seconds}, indent=2), encoding="utf-8")
elif runtime_file.exists():
    mljar_seconds = json.loads(runtime_file.read_text(encoding="utf-8"))["first_full_run_seconds"]
else:
    mljar_seconds = mljar_observed_seconds
mljar_pred = np.asarray(automl.predict(X_test))
mljar_prob = np.asarray(automl.predict_proba(X_test)) if TASK == "classification" else None
try:
    automl.report_structured(format="json")
except Exception as exc:
    print("结构化报告显式调用失败:", type(exc).__name__, exc)
mljar_best_name = type(getattr(automl, "_best_model", None)).__name__
print("MLJAR 最佳模型:", mljar_best_name)
print("MLJAR 运行秒数:", round(mljar_seconds, 2))


AutoML directory: C:\Users\<user>\Documents\ML26-qyr-develop\benchmark-output\wine\mljar
The task is multiclass_classification with evaluation metric logloss
AutoML will use algorithms: ['Random Forest', 'Xgboost', 'LightGBM', 'CatBoost']
AutoML is generating and selecting models automatically. Review outputs before important use.
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'ensemble']
Skip simple_algorithms because no parameters were generated.
* Step default_algorithms will try to check up to 4 models


1_Default_LightGBM logloss 0.011968 trained in 14.48 seconds


2_Default_Xgboost logloss 0.075315 trained in 8.1 seconds


3_Default_CatBoost logloss 0.004079 trained in 7.33 seconds


4_Default_RandomForest logloss 0.200184 trained in 1.82 seconds


* Step ensemble will try to check up to 1 model


Ensemble logloss 0.004079 trained in 0.12 seconds


AutoML fit time: 36.42 seconds
AutoML best model: 3_Default_CatBoost
MLJAR 最佳模型: ModelFramework
MLJAR 运行秒数: 36.42


## 4. 外部测试结果、工件盘点与 HTML/TeX 报告

In [5]:
from fnmatch import fnmatch
from html import escape

def classification_metrics(name, pred, prob):
    row = {"工具": name, "Accuracy": accuracy_score(y_test, pred),
           "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
           "Macro-F1": f1_score(y_test, pred, average="macro"),
           "混淆矩阵": confusion_matrix(y_test, pred).tolist()}
    try:
        if y.nunique() == 2:
            score = prob[:, 1] if prob.ndim == 2 else prob
            row["ROC-AUC"] = roc_auc_score(y_test, score)
        else:
            row["ROC-AUC"] = roc_auc_score(y_test, prob, multi_class="ovr", average="macro")
    except Exception:
        row["ROC-AUC"] = np.nan
    return row

def regression_metrics(name, pred):
    return {"工具": name, "MAE": mean_absolute_error(y_test, pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))), "R2": r2_score(y_test, pred)}

if TASK == "classification":
    metrics = pd.DataFrame([classification_metrics("TabularX", tbx_pred, tbx_prob),
                            classification_metrics("MLJAR", mljar_pred, mljar_prob)])
else:
    metrics = pd.DataFrame([regression_metrics("TabularX", tbx_pred),
                            regression_metrics("MLJAR", mljar_pred)])
display(metrics)
metrics.to_json(OUTPUT / "external_test_metrics.json", orient="records", force_ascii=False, indent=2)

def matching(root, patterns):
    root = Path(root)
    return sorted(str(p.relative_to(root)) for p in root.rglob("*")
                  if p.is_file() and any(fnmatch(p.name.lower(), pat.lower()) for pat in patterns))

checks = [
    ("最佳模型及可重载文件", ["pipeline.joblib"], ["*.lightgbm", "*.catboost", "*.random_forest", "*.xgboost.json", "ensemble.json"]),
    ("候选模型与验证分数", ["model_scores.json"], ["leaderboard.csv", "params.json"]),
    ("预处理记录", ["preprocess.html"], ["README.md"]),
    ("特征重要性", ["*shap_importance*"], ["*importance*"]),
    ("SHAP", ["*shap_summary*"], ["*shap_summary*"]),
    ("ALE", ["ale*.png", "*_ale_*.png"], ["ale*.png", "*_ale_*.png"]),
    ("学习曲线或调参轨迹", ["*trial_trajectory*"], ["*learning_curve*"]),
    ("混淆矩阵", ["*confusion_matrix*"], ["*confusion_matrix*"]),
    ("公平性", ["fairness.json"], ["*fairness*"]),
    ("泄露、划分与评估协议审计", ["audit.json", "model_search_audit.json"], ["*leakage*", "*audit*"]),
    ("漂移与群体覆盖", ["audit.json"], ["*drift*", "*coverage*"]),
    ("重载一致性", ["reload_check.json"], ["*reload*"]),
    ("交付状态与失败原因", ["delivery.json"], ["*delivery*"]),
    ("原生 HTML 报告", ["report.html"], ["*.html"]),
]
rows = []
for label, tp, mp in checks:
    tf, mf = matching(tbx_run_dir, tp), matching(mljar_dir, mp)
    rows.append({"工件": label, "TabularX": "生成并核验" if tf else ("不适用" if label == "公平性" and not sensitive_columns else "未观测"),
                 "MLJAR": "生成并核验" if mf else ("不适用" if label == "公平性" and not sensitive_columns else "未观测"),
                 "TabularX 证据": "；".join(tf[:2]), "MLJAR 证据": "；".join(mf[:2])})
artifact_matrix = pd.DataFrame(rows)
display(artifact_matrix)
artifact_matrix.to_csv(OUTPUT / "artifact_matrix.csv", index=False, encoding="utf-8-sig")

tbx_best = str(ae.result.searched.best_model_name)
tbx_advantage = "将测试隔离、可信审计、模型重载核验和交付状态纳入同一次运行。"
tbx_limit = "本次未观察到 ALE；少量算法试验若失败会保留异常记录，不据此掩盖运行边界。"
mljar_advantage = "Explain 模式提供逐模型 Markdown、学习曲线、混淆矩阵和特征重要性。"
mljar_limit = "本次未观察到可信审计、重载一致性、交付状态和原生 HTML；SHAP 覆盖以实际文件为准。"

html = f'''<!doctype html><html lang="zh-CN"><meta charset="utf-8"><style>
body{{font-family:"Microsoft YaHei",sans-serif;max-width:1120px;margin:32px auto;line-height:1.65;color:#243042}}
table{{border-collapse:collapse;width:100%;font-size:13px}}th,td{{border:1px solid #ccd5e0;padding:7px;vertical-align:top}}th{{background:#edf3f8}}
.note{{padding:12px;background:#fff8dc;border-left:4px solid #d9a400}}</style>
<h1>MLJAR 与 TabularX 完整工具包对比</h1>
<p><strong>数据集：{escape(DATASET_TITLE)}（{escape(SOURCE)}，{escape(TASK_CN)}）</strong>。{escape(DESCRIPTION)}</p>
<p class="note">双方使用相同开发集和外部测试集；性能只描述本次固定配置，核心比较对象是实际生成的工件。</p>
<h2>1. 外部测试集结果</h2>{metrics.to_html(index=False, float_format=lambda x: f"{x:.4f}")}
<h2>2. 工件盘点</h2>{artifact_matrix.to_html(index=False, escape=True)}
<h2>3. 对比结果</h2><ul>
<li><strong>TabularX：</strong>优点：{tbx_advantage}不足：{tbx_limit}</li>
<li><strong>MLJAR：</strong>优点：{mljar_advantage}不足：{mljar_limit}</li>
<li>TabularX 最佳模型：{escape(tbx_best)}；MLJAR 最佳模型：{escape(mljar_best_name)}。</li>
</ul><h2>4. 运行信息</h2><p>TabularX：{tbx_seconds:.1f} 秒；MLJAR：{mljar_seconds:.1f} 秒；随机种子 {SEED}。</p></html>'''
html_path = OUTPUT / "comparison_report.html"
html_path.write_text(html, encoding="utf-8")

def latex_escape(value):
    text = str(value)
    for old, new in [("\\", r"\textbackslash{}"), ("_", r"\_"), ("%", r"\%"), ("&", r"\&"), ("#", r"\#")]:
        text = text.replace(old, new)
    return text

metric_table = metrics.drop(columns=["混淆矩阵"], errors="ignore").to_latex(index=False, float_format="%.4f")
artifact_short = artifact_matrix[["工件", "TabularX", "MLJAR"]].to_latex(index=False)
tex_template = r'''% !TeX encoding = UTF-8
% !TeX program = xelatex
\documentclass[degree=bachelor,language=chinese,fontset=fandol]{thuthesis}
\usepackage{booktabs,longtable,array,hyperref}
\begin{document}
\begin{titlepage}\centering\vspace*{2.3cm}
{\zihao{1}\bfseries MLJAR 与 TabularX 完整工具包对比\par}
\vspace{1.2cm}{\zihao{3}@@DATASET@@ 数据集实验报告\par}
\vfill{\zihao{-4}TabularX 课程项目\quad 2026 年 9 月 29 日\par}
\end{titlepage}
\frontmatter\tableofcontents\mainmatter
\chapter{实验概述}
\textbf{数据集：@@DATASET@@（@@SOURCE@@，@@TASK@@）。}@@DESCRIPTION@@

双方使用相同开发集和外部测试集、随机种子 42，以及 Random Forest、XGBoost、LightGBM、CatBoost。外部测试集只在最终模型锁定后评估一次。
\chapter{外部测试集结果}
@@METRICS@@
\chapter{工件盘点}
@@ARTIFACTS@@
“未观测”仅表示本次运行目录中没有找到对应工件，不外推为软件永远不支持。
\chapter{工具优缺点}
\textbf{TabularX 优点：}@@TBX_ADV@@

\textbf{TabularX 不足：}@@TBX_LIMIT@@

\textbf{MLJAR 优点：}@@MLJAR_ADV@@

\textbf{MLJAR 不足：}@@MLJAR_LIMIT@@
\chapter{结论}
本实验比较固定配置下的完整工具包输出，不是严格等预算的性能竞赛。后续数据集复用相同的数据隔离、工件状态词和解释边界。
\end{document}
'''
tex = (tex_template.replace("@@DATASET@@", latex_escape(DATASET_TITLE))
       .replace("@@SOURCE@@", latex_escape(SOURCE)).replace("@@TASK@@", latex_escape(TASK_CN))
       .replace("@@DESCRIPTION@@", latex_escape(DESCRIPTION)).replace("@@METRICS@@", metric_table)
       .replace("@@ARTIFACTS@@", artifact_short).replace("@@TBX_ADV@@", latex_escape(tbx_advantage))
       .replace("@@TBX_LIMIT@@", latex_escape(tbx_limit)).replace("@@MLJAR_ADV@@", latex_escape(mljar_advantage))
       .replace("@@MLJAR_LIMIT@@", latex_escape(mljar_limit)))
tex_path = OUTPUT / f"{DATASET_TITLE}.tex"
tex_path.write_text(tex, encoding="utf-8")
print("HTML:", html_path)
print("TeX:", tex_path)

# 使用与 Bank Marketing 一致的 THUThesis 章节与表格格式重写 TeX。
thuthesis_helper = PROJECT_ROOT / "introduction" / "benchmarks" / "thuthesis_report.py"
subprocess.run([sys.executable, str(thuthesis_helper), DATASET_KEY], check=True)
print("统一格式 TeX:", tex_path)


,工具,Accuracy,Balanced Accuracy,Macro-F1,混淆矩阵,ROC-AUC
0,TabularX,0.972222,0.97619,0.974321,"[[12, 0, 0], [1, 13, 0], [0, 0, 10]]",1.0
1,MLJAR,1.000000,1.00000,1.000000,"[[12, 0, 0], [0, 14, 0], [0, 0, 10]]",1.0


,工件,TabularX,MLJAR,TabularX 证据,MLJAR 证据
0,最佳模型及可重载文件,生成并核验,生成并核验,pipeline.joblib,1_Default_LightGBM\learner_fold_0.lightgbm；2_D...
1,候选模型与验证分数,生成并核验,生成并核验,evidence\model_scores.json,leaderboard.csv；params.json
2,预处理记录,生成并核验,生成并核验,reports\preprocess.html,1_Default_LightGBM\README.md；2_Default_Xgboost...
3,特征重要性,生成并核验,生成并核验,reports\assets\random_forest_shap_importance.png,1_Default_LightGBM\learner_fold_0_importance.c...
4,SHAP,生成并核验,生成并核验,reports\assets\random_forest_shap_summary.png,1_Default_LightGBM\learner_fold_0_shap_summary...
5,ALE,未观测,未观测,,
6,学习曲线或调参轨迹,生成并核验,生成并核验,reports\assets\catboost_trial_trajectory.png；r...,1_Default_LightGBM\learning_curves.png；2_Defau...
7,混淆矩阵,生成并核验,生成并核验,reports\assets\random_forest_confusion_matrix.png,1_Default_LightGBM\confusion_matrix.png；1_Defa...
8,公平性,不适用,不适用,,
9,泄露、划分与评估协议审计,生成并核验,未观测,evidence\audit.json；evidence\model_search_audi...,


HTML: C:\Users\<user>\Documents\ML26-qyr-develop\benchmark-output\wine\comparison_report.html
TeX: C:\Users\<user>\Documents\ML26-qyr-develop\benchmark-output\wine\comparison_report_thuthesis.tex


## 结论边界

- 本实验不是严格等预算的性能竞赛。
- `未观测` 不等于软件在任何配置下都不支持。
- HTML 与 TeX 汇总报告由本 Notebook 基于实际工件自动生成。